# 05 · Dịch máy (Anh ↔ Việt)

**Các bước:**
1. Đọc dữ liệu
2. Làm sạch
3. Thử vài model có sẵn → chọn model tốt nhất
4. Fine-tune model đó
5. Chọn cách sinh (số beam)
6. Dịch tập test → nộp bài

Lý thuyết: xem `knowledge/05_dich_may.md`

In [ ]:
# !pip install -U transformers datasets accelerate sacrebleu sentencepiece

## 0. Import

In [ ]:
import re
import unicodedata

import pandas as pd
import torch
import sacrebleu
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 1. Cấu hình
Chỉ cần sửa ô này khi thi.

In [ ]:
SRC_LANG = "en"     # ngôn ngữ nguồn
TGT_LANG = "vi"     # ngôn ngữ đích

TRAIN_FILE = None   # vd "train.csv"   (None = dùng dữ liệu mẫu)
VALID_FILE = None   # vd "valid.csv"
TEST_FILE = None    # vd "test.csv"

SRC_COL = "src"     # tên cột câu nguồn trong file
TGT_COL = "tgt"     # tên cột câu đích trong file

MAX_LEN = 256

## 2. Đọc dữ liệu

In [ ]:
def load_demo_data():
    data = load_dataset("Helsinki-NLP/opus-100", "en-vi")

    train_rows = data["train"].select(range(20000))["translation"]
    valid_rows = data["validation"].select(range(500))["translation"]

    train = pd.DataFrame({
        "src": [row[SRC_LANG] for row in train_rows],
        "tgt": [row[TGT_LANG] for row in train_rows],
    })
    valid = pd.DataFrame({
        "src": [row[SRC_LANG] for row in valid_rows],
        "tgt": [row[TGT_LANG] for row in valid_rows],
    })
    return train, valid

In [ ]:
if TRAIN_FILE is None:
    train_df, valid_df = load_demo_data()
else:
    train_df = pd.read_csv(TRAIN_FILE).rename(columns={SRC_COL: "src", TGT_COL: "tgt"})
    valid_df = pd.read_csv(VALID_FILE).rename(columns={SRC_COL: "src", TGT_COL: "tgt"})

print(len(train_df), "câu train")
print(len(valid_df), "câu valid")
train_df.head()

## 3. Làm sạch dữ liệu train
- Chuẩn hoá Unicode (rất quan trọng với tiếng Việt)
- Bỏ dòng trống, dòng trùng
- Bỏ cặp câu lệch độ dài quá nhiều (thường là lỗi ghép câu)

⚠️ Chỉ làm sạch **train**, không đụng vào valid/test.

In [ ]:
def normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_data(df):
    df = df.dropna().copy()
    df["src"] = df["src"].apply(normalize)
    df["tgt"] = df["tgt"].apply(normalize)

    df = df[df["src"] != ""]
    df = df[df["tgt"] != ""]
    df = df.drop_duplicates()

    src_len = df["src"].str.split().str.len()
    tgt_len = df["tgt"].str.split().str.len()
    ratio = src_len / tgt_len
    df = df[(ratio > 0.33) & (ratio < 3)]

    return df.reset_index(drop=True)

In [ ]:
print("trước:", len(train_df))
train_df = clean_data(train_df)
print("sau  :", len(train_df))

valid_df["src"] = valid_df["src"].apply(normalize)
valid_df["tgt"] = valid_df["tgt"].apply(normalize)

## 4. Hàm load model và dịch

| Model | Ghi chú |
|---|---|
| `Helsinki-NLP/opus-mt-en-vi` | nhỏ, nhanh, chỉ 1 chiều |
| `VietAI/envit5-translation` | mạnh cho Anh–Việt, cần thêm `"en: "` trước câu |
| `facebook/nllb-200-distilled-600M` | đa ngữ, cần mã ngôn ngữ |

In [ ]:
NLLB_CODES = {"en": "eng_Latn", "vi": "vie_Latn"}


def load_model(name):
    if "nllb" in name:
        tokenizer = AutoTokenizer.from_pretrained(name, src_lang=NLLB_CODES[SRC_LANG], tgt_lang=NLLB_CODES[TGT_LANG])
    else:
        tokenizer = AutoTokenizer.from_pretrained(name)

    model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)

    # NLLB: ép từ đầu tiên của câu dịch là mã ngôn ngữ đích
    if "nllb" in name:
        target_code = NLLB_CODES[TGT_LANG]
        model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_code)

    return tokenizer, model


def get_prefix(name):
    # EnViT5 cần "en: " hoặc "vi: " ở đầu câu nguồn
    if "envit5" in name:
        return SRC_LANG + ": "
    return ""

In [ ]:
def clean_output(text):
    text = re.sub(r"^(vi|en):\s*", "", text)      # EnViT5 trả về "vi: ..." -> bỏ "vi: "
    return text.strip()


def translate(texts, tokenizer, model, prefix="", num_beams=4, batch_size=32):
    model.eval()
    results = []

    for start in range(0, len(texts), batch_size):
        batch = [prefix + t for t in texts[start:start + batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN)
        inputs = inputs.to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, num_beams=num_beams, max_new_tokens=MAX_LEN)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(clean_output(t) for t in decoded)

    return results

In [ ]:
def score(predictions, references):
    bleu = sacrebleu.corpus_bleu(predictions, [references]).score
    chrf = sacrebleu.corpus_chrf(predictions, [references], word_order=2).score
    return {"BLEU": round(bleu, 2), "chrF++": round(chrf, 2)}

## 5. So sánh các model có sẵn (chưa fine-tune)
Dùng 200 câu valid cho nhanh.

In [ ]:
CANDIDATES = [
    "Helsinki-NLP/opus-mt-en-vi",
    "VietAI/envit5-translation",
    "facebook/nllb-200-distilled-600M",
]

sample = valid_df.head(200)
results = {}

for name in CANDIDATES:
    tokenizer, model = load_model(name)
    predictions = translate(sample["src"].tolist(), tokenizer, model, prefix=get_prefix(name))
    results[name] = score(predictions, sample["tgt"].tolist())
    print(name, results[name])

    del model
    torch.cuda.empty_cache()

In [ ]:
best_name = max(results, key=lambda name: results[name]["BLEU"])
print("Model tốt nhất:", best_name)

## 6. Fine-tune model tốt nhất

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

MODEL_NAME = best_name
PREFIX = get_prefix(MODEL_NAME)

tokenizer, model = load_model(MODEL_NAME)

Chuyển câu thành số. `labels` là câu đích (đã token hoá).

In [ ]:
def tokenize(batch):
    sources = [PREFIX + s for s in batch["src"]]
    targets = batch["tgt"]
    if "envit5" in MODEL_NAME:
        targets = [TGT_LANG + ": " + t for t in targets]

    inputs = tokenizer(sources, max_length=MAX_LEN, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_LEN, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs


train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True, remove_columns=["src", "tgt"])
valid_ds = Dataset.from_pandas(valid_df).map(tokenize, batched=True, remove_columns=["src", "tgt"])

Hàm tính BLEU trong lúc train:

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    pred_ids, label_ids = eval_pred
    pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)   # -100 -> pad

    predictions = [clean_output(t) for t in tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    references = [clean_output(t) for t in tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return score(predictions, references)

Tham số train:
- `learning_rate`: 5e-5 cho Marian/NLLB, 3e-4 cho T5 (EnViT5)
- `label_smoothing_factor=0.1`: chuẩn trong dịch máy
- Giữ checkpoint có BLEU cao nhất

In [ ]:
is_t5 = "t5" in MODEL_NAME

args = Seq2SeqTrainingArguments(
    output_dir="mt_output",
    learning_rate=3e-4 if is_t5 else 5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    warmup_ratio=0.05,
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="BLEU",
    predict_with_generate=True,
    generation_num_beams=4,
    generation_max_length=MAX_LEN,
    fp16=torch.cuda.is_available() and not is_t5,    # T5 dùng fp16 dễ bị lỗi NaN
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
trainer.save_model("mt_best")
tokenizer.save_pretrained("mt_best")

## 7. Chọn số beam tốt nhất

In [ ]:
sample = valid_df.head(200)

for beams in [1, 4, 6]:
    predictions = translate(sample["src"].tolist(), tokenizer, model, prefix=PREFIX, num_beams=beams)
    print("beam =", beams, score(predictions, sample["tgt"].tolist()))

In [ ]:
BEST_BEAMS = 4   # sửa theo kết quả ô trên

## 8. Dịch tập test và lưu bài nộp

In [ ]:
if TEST_FILE is not None:
    test_df = pd.read_csv(TEST_FILE).rename(columns={SRC_COL: "src"})
    test_df["src"] = test_df["src"].apply(normalize)

    test_df["prediction"] = translate(test_df["src"].tolist(), tokenizer, model, prefix=PREFIX, num_beams=BEST_BEAMS)
    test_df.to_csv("submission.csv", index=False)
    print(test_df.head())

---
## 9. Nâng cao (làm nếu còn thời gian)

**MBR decoding:** sinh nhiều bản dịch cho mỗi câu, chọn bản **giống các bản còn lại nhất**.
Thường tăng 0.5–1.5 BLEU.

In [ ]:
from sacrebleu.metrics import CHRF

chrf_metric = CHRF(word_order=2)


def mbr_choose(candidates):
    best_text = candidates[0]
    best_score = -1
    for cand in candidates:
        total = 0
        for other in candidates:
            if other != cand:
                total += chrf_metric.sentence_score(cand, [other]).score
        if total > best_score:
            best_text = cand
            best_score = total
    return best_text


def translate_mbr(texts, tokenizer, model, prefix="", num_candidates=8):
    model.eval()
    results = []
    for text in texts:
        inputs = tokenizer([prefix + text], return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=num_candidates,
                num_return_sequences=num_candidates,
                max_new_tokens=MAX_LEN,
            )
        candidates = [clean_output(t) for t in tokenizer.batch_decode(outputs, skip_special_tokens=True)]
        results.append(mbr_choose(candidates))
    return results

In [ ]:
sample = valid_df.head(100)
predictions = translate_mbr(sample["src"].tolist(), tokenizer, model, prefix=PREFIX)
print("MBR:", score(predictions, sample["tgt"].tolist()))

**Các mẹo khác:**
- Câu test quá dài → tách thành từng câu, dịch rồi ghép lại.
- Back-translation: dùng model chiều ngược (vd `opus-mt-vi-en`) dịch văn bản tiếng Việt → tạo thêm cặp câu giả để train.
- Ensemble: lấy ứng viên từ 2–3 model khác nhau, gộp lại rồi chọn bằng `mbr_choose`.